<a href="https://colab.research.google.com/github/Prasanna-Mahajan-2006/Deepfake-Detector/blob/main/DSP_TRANFER_LEARNING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kagglehub

In [ ]:
#importing necessary libraries

import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import Sequence
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
!pip install kagglehub -q

import kagglehub

# downloading the dataset


print("Downloading dataset via kagglehub...")
# This will download the dataset and save the exact folder location to the 'path' variable
path = kagglehub.dataset_download("manjilkarki/deepfake-and-real-images")

print(f"Success! Dataset downloaded to: {path}")

In [ ]:
# Prasanna - changing the above code as Kaggle has updated its library so we no longer need the API keys

# 1. Define Variables (To prevent NameErrors)
IMG_SIZE = (224, 224)
BATCH_SIZE = 64

# 2. Download the dataset effortlessly (No API keys or Unzipping needed!)
print("Downloading dataset via kagglehub...")
path = kagglehub.dataset_download("manjilkarki/deepfake-and-real-images")
print(f"Dataset securely downloaded to: {path}\n")

# 3. Load the datasets into Keras
print("Loading Training Data...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("\nLoading Validation Data...")
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("\nLoading Testing Data...")
test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("\nSuccess! All datasets are loaded and ready for training.")

In [ ]:
# Prasanna - Now trying with Transfer Learning methods - first train the frozen model for 5 epochs, then unfreeze some layers in phase 2 and check for improvements
# ==========================================
# 1. BUILD THE MODEL (PHASE 1 SETUP)
# ==========================================
print("Building the architecture...")
IMG_SIZE = (224, 224)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
], name="data_augmentation")

base_model = keras.applications.MobileNetV3Small(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
# Freeze the base model for the warm-up phase
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid', kernel_regularizer=regularizers.l2(0.01))(x)

model = keras.Model(inputs, outputs)

# Compile for Phase 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

# ==========================================
# 2. PHASE 1: WARM-UP TRAINING
# ==========================================
print("\n--- PHASE 1: Warming up the classification head (5 Epochs) ---")
# We just want to stabilize the new dense layers quickly
history_warmup = model.fit(
    train_ds,
    epochs=5,
    validation_data=val_ds
)

# ==========================================
# 3. PHASE 2: FINE-TUNING THE BASE MODEL
# ==========================================
print("\n--- PHASE 2: Unfreezing MobileNet for Fine-Tuning ---")

# Unfreeze the base model
base_model.trainable = True

# MobileNetV3Small has about 229 layers. We will freeze the bottom 150 layers
# (which detect basic lines/colors) and unfreeze the top 79 layers
# (which detect complex shapes like facial artifacts).
print(f"Total layers in base model: {len(base_model.layers)}")
for layer in base_model.layers[:150]:
    layer.trainable = False

# CRUCIAL: Recompile the model with a severely reduced learning rate (1e-5 instead of 1e-3)
# We want tiny, gentle updates to the pre-trained weights.
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

# Setup Advanced Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint("best_finetuned_model.keras", save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    # This automatically lowers the learning rate if validation loss plateaus for 2 epochs
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1)
]

print("Starting deep fine-tuning... (Aiming for 85%+)")
history_finetune = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=callbacks
)

# ==========================================
# 4. FINAL EVALUATION
# ==========================================
print("\n--- Generating Final Evaluation Metrics on Test Set ---")
test_loss, test_acc, test_prec, test_rec = model.evaluate(test_ds)
print(f"\nFINAL Test Accuracy: {test_acc*100:.2f}%")

print("Re-loading test data for Confusion Matrix (shuffle=False)...")
# Note: Ensure 'path' variable is still in memory from your kagglehub download cell
test_ds_unshuffled = tf.keras.utils.image_dataset_from_directory(
    f"{path}/Dataset/Test",
    image_size=IMG_SIZE,
    batch_size=64,
    shuffle=False
)

y_true = np.concatenate([y for x, y in test_ds_unshuffled], axis=0)
y_pred_probs = model.predict(test_ds_unshuffled)
y_pred = (y_pred_probs > 0.5).astype("int32")

print("\n--- Detailed Classification Report ---")
print(classification_report(y_true, y_pred, target_names=['Real (0)', 'Fake (1)']))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('Fine-Tuned Deepfake Detection Performance')
plt.show()